
# Patient Cohort Exploratory Analysis

This notebook performs exploratory analysis on the synthetic pharma dataset used for the commercial analytics dashboard.

We investigate several metrics:

- Distribution of time to fill by payer and channel type.
- Distribution of prior authorization outcomes.
- Average medication cost per therapeutic class.
- New patient starts over time.


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set(style='whitegrid')

# Path to data directory
base_dir = '../data'

# Load the fact and dimension tables
rx = pd.read_csv(f'{base_dir}/fact_rx_fills.csv', parse_dates=['prescription_date','fill_date'])
payers = pd.read_csv(f'{base_dir}/dim_payers.csv')
pharmacies = pd.read_csv(f'{base_dir}/dim_pharmacies.csv')
drugs = pd.read_csv(f'{base_dir}/dim_drugs.csv')


In [ ]:

# Merge payer information
rx_pay = rx.merge(payers, on='payer_id')

# Boxplot of time to fill by payer
plt.figure(figsize=(8,4))
sns.boxplot(x='payer_name', y='time_to_fill_days', data=rx_pay)
plt.title('Time to Fill Distribution by Payer')
plt.xlabel('Payer')
plt.ylabel('Time to fill (days)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:

# Count fills by channel type
channel_counts = rx.groupby('channel_type').size().reset_index(name='fills')

# Bar chart
plt.figure(figsize=(6,4))
sns.barplot(x='channel_type', y='fills', data=channel_counts)
plt.title('Fills by Channel Type')
plt.xlabel('Channel')
plt.ylabel('Number of Fills')
plt.tight_layout()
plt.show()


In [ ]:

pa_data = rx[rx['prior_auth_required']==True]
pa_counts = pa_data['prior_auth_status'].value_counts().reset_index()
pa_counts.columns = ['status','count']

plt.figure(figsize=(6,4))
sns.barplot(x='status', y='count', data=pa_counts)
plt.title('Prior Authorization Outcomes')
plt.xlabel('Prior Auth Status')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:

# Merge drugs
rx_drug = rx.merge(drugs[['drug_id','therapeutic_class']], on='drug_id')
avg_cost = rx_drug.groupby('therapeutic_class')['cost'].mean().reset_index()

plt.figure(figsize=(8,4))
sns.barplot(x='therapeutic_class', y='cost', data=avg_cost)
plt.title('Average Fill Cost by Therapeutic Class')
plt.xlabel('Therapeutic Class')
plt.ylabel('Average Cost ($)')
plt.tight_layout()
plt.show()


In [ ]:

# Filter new starts
new_starts = rx[rx['is_new_start']==True]
# Monthly counts
monthly_new = new_starts.resample('M', on='prescription_date').size().reset_index(name='new_starts')

plt.figure(figsize=(8,4))
plt.plot(monthly_new['prescription_date'], monthly_new['new_starts'], marker='o')
plt.title('Monthly New Patient Starts')
plt.xlabel('Month')
plt.ylabel('New Starts')
plt.tight_layout()
plt.show()
